## Step 1 — Load the Cleaned Genome-Level Dataset

The cleaned dataset generated during data curation is loaded as the input for
genome sequence retrieval.

The dataset contains one unique record per genome and a binary ciprofloxacin
resistance label:

- Susceptible → 0
- Resistant → 1

The Genome ID is used as the identifier for retrieving the corresponding genome
from BV-BRC.

In [1]:
import pandas as pd

genomes = pd.read_csv(
    "../data/processed/ciprofloxacin_genome_level.csv"
)

print("Dataset shape:", genomes.shape)

print("\nFirst 5 rows:")
print(genomes.head())

print("\nLabel distribution:")
print(genomes["Label"].value_counts().sort_index())

Dataset shape: (1205, 3)

First 5 rows:
    Genome ID         Phenotype  Label
0  1280.15865  ('Susceptible',)      0
1  1280.15866  ('Susceptible',)      0
2  1280.15867  ('Susceptible',)      0
3  1280.15868  ('Susceptible',)      0
4  1280.15869  ('Susceptible',)      0

Label distribution:
Label
0    691
1    514
Name: count, dtype: int64


In [2]:
genomes["Genome ID"] = genomes["Genome ID"].astype(str)

In [3]:
print(genomes["Genome ID"].head())
print(type(genomes.loc[0, "Genome ID"]))

0    1280.15865
1    1280.15866
2    1280.15867
3    1280.15868
4    1280.15869
Name: Genome ID, dtype: str
<class 'str'>


## Step 2 — Select a Representative Genome

Before attempting bulk sequence retrieval, a single genome is selected to
validate the complete retrieval workflow.

Genome ID `1280.15865` is used as the test genome.

In [4]:
genome_id = genomes.loc[0, "Genome ID"]

print("Testing Genome ID:", genome_id)

Testing Genome ID: 1280.15865


In [5]:
%pip install requests


   ---------------------------------------- 0/5 [urllib3]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------------------------------- 5/5 [requests]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import requests

print(requests.__version__)

2.34.2


## Step 3 — Validate the Genome ID Against BV-BRC

The selected Genome ID is queried against the BV-BRC Genome API to confirm that
the identifier corresponds to a publicly available genome record.

The returned metadata is inspected to verify the organism, strain, genome
length, assembly information, and genome quality before sequence retrieval.

In [7]:
import requests

url = f"https://www.bv-brc.org/api/genome/{genome_id}"

response = requests.get(url, timeout=30)

print("HTTP status:", response.status_code)

response.raise_for_status()

genome_record = response.json()

print(genome_record)

HTTP status: 200
{'antimicrobial_resistance_evidence': 'AMR Panel', 'assembly_accession': 'GCA_002904355.1', 'assembly_method': 'CLC NGS Cell v. 8.0', 'bioproject_accession': 'PRJNA295807', 'biosample_accession': 'SAMN04089946', 'body_sample_site': 'nose', 'cds': 2583, 'cds_ratio': 0.95, 'cell_shape': '', 'checkm_completeness': 100, 'checkm_contamination': 0, 'class': 'Bacilli', 'coarse_consistency': 100, 'collection_date': '2014-2015', 'collection_year': 2015, 'common_name': 'Staphylococcus_aureus', 'completion_date': '2018-01-29T00:00:00Z', 'contig_l50': 5, 'contig_n50': 198837, 'contigs': 26, 'date_inserted': '2018-05-25T20:40:11.983Z', 'date_modified': '2018-05-25T20:40:11.983Z', 'family': 'Staphylococcaceae', 'fine_consistency': 100, 'gc_content': 32.700172, 'genbank_accessions': 'LKNK00000000', 'genome_id': '1280.15865', 'genome_length': 2730771, 'genome_name': 'Staphylococcus aureus strain N32_250215', 'genome_quality': 'Good', 'genome_status': 'WGS', 'genus': 'Staphylococcus', 

## Step 4 — Retrieve the Genome Sequence in FASTA Format

The BV-BRC Genome Sequence API is queried using the validated Genome ID.

The nucleotide sequence is requested in FASTA format. Because bacterial genomes
may consist of multiple contigs, the returned FASTA may contain multiple
sequence records.

In [8]:
fasta_url = (
    f"https://www.bv-brc.org/api/genome_sequence/"
    f"?eq(genome_id,{genome_id})"
    f"&http_accept=application/dna+fasta"
)

fasta_response = requests.get(fasta_url, timeout=60)

print("HTTP status:", fasta_response.status_code)

fasta_response.raise_for_status()

fasta_text = fasta_response.text

print("\nFASTA preview:")
print(fasta_text[:1000])

HTTP status: 200

FASTA preview:
>accn|LKNK01000018   LKNK01000018.1   [Staphylococcus aureus strain N32_250215 | 1280.15865]
tcttactgctgtttttttagggatttatgtcccagcctctttttggttttaattttacaa
taattgcttagtatcttttaaatcaaatgtgcaattgttaatatagctctatattagagt
gttttggatgaaacatttgctatgataaggtaaaattaaattaagaaagtgctttaatgc
tttaactgtataaatcatttaaatagttttgctacaacgaatatgttattataagttacc
aaaatattttagagaaagaaatcaaacaaataaattgtattttgtttgatgctttaatac
aatgtggatactattcatcgactgtaaggagtgagatgatatgagaatagaacgagtaga
tgatacaactgtaaaattgtttataacatatagcgatatcgaggcccgtggatttagtcg
tgaagatttatggacaaatcgcaaacgtggcgaagaattcttttggtcaatgatggatga
aattaacgaagaagaagattttgttgtagaaggtccattatggattcaagtacatgcctt
tgaaaaaggtgtcgaagtcacaatttctaaatctaaaaatgaagatatgatgaatatgtc
tgatgatgatgcaactgatcaatttgatgaacaagttcaagaattgttagctcaaacatt
agaaggtgaagatcaattagaagaattattcgagcaacgaacaaaagaaaaagaagctca
aggttctaaacgtcaaaagtcttcagcacgtaaaaatacaagaacaattattgtgaaatt
taacgatttagaagatgttattaattatgcatatcatagcaatccaataactacagagtt
tgaagatttgttatatatgg

## Step 5 — Save the Retrieved FASTA

The retrieved nucleotide sequence is saved locally in the project's genome
data directory.

Each FASTA file is named using its corresponding BV-BRC Genome ID so that the
sequence can be unambiguously linked back to the phenotype dataset.

Output location:

`data/genomes/<Genome_ID>.fna`

In [9]:
from pathlib import Path

genome_dir = Path("../data/genomes")
genome_dir.mkdir(parents=True, exist_ok=True)

fasta_path = genome_dir / f"{genome_id}.fna"

with open(fasta_path, "w") as f:
    f.write(fasta_text)

print("FASTA saved to:", fasta_path)
print("File size:", fasta_path.stat().st_size, "bytes")

FASTA saved to: ..\data\genomes\1280.15865.fna
File size: 2805259 bytes


## Step 6 — Validate the Retrieved FASTA

The saved FASTA file is inspected to determine:

- Number of FASTA sequence records
- Total nucleotide sequence length
- Genome length reported by BV-BRC
- Difference between retrieved and reported genome length

This validation is performed before bulk retrieval to ensure that the sequence
download process is behaving as expected and to identify any discrepancies
that must be understood before processing the complete genome cohort.

In [10]:
# Validate the saved FASTA

with open(fasta_path, "r") as f:
    lines = f.readlines()

header_count = sum(line.startswith(">") for line in lines)

sequence = "".join(
    line.strip()
    for line in lines
    if not line.startswith(">")
)

print("Number of FASTA records:", header_count)
print("Total sequence length:", len(sequence))
print("Expected genome length:", genome_record["genome_length"])
print("Length difference:", len(sequence) - genome_record["genome_length"])

Number of FASTA records: 25
Total sequence length: 2712467
Expected genome length: 2730771
Length difference: -18304


## Step 7 — Investigate Individual Sequence Records

The individual genome sequence records associated with the test Genome ID are
retrieved in JSON format.

Each sequence record is inspected for its identifier, sequence type, accession,
and length. This allows differences between the genome-level metadata and the
retrieved FASTA records to be investigated before implementing bulk sequence
retrieval.

In [11]:
sequence_url = (
    f"https://www.bv-brc.org/api/genome_sequence/"
    f"?eq(genome_id,{genome_id})"
    f"&http_accept=application/json"
)

sequence_response = requests.get(sequence_url, timeout=60)

print("HTTP status:", sequence_response.status_code)

sequence_response.raise_for_status()

sequence_records = sequence_response.json()

print("Number of sequence records:", len(sequence_records))

HTTP status: 200
Number of sequence records: 25


In [12]:
for record in sequence_records:
    print(
        record["sequence_id"],
        "|",
        record["sequence_type"],
        "|",
        record["length"],
        "bp",
        "|",
        record.get("accession")
    )

1280.15865.con.0010 | contig | 87177 bp | LKNK01000018
1280.15865.con.0020 | contig | 198837 bp | LKNK01000003
1280.15865.con.0008 | contig | 83819 bp | LKNK01000016
1280.15865.con.0002 | contig | 130579 bp | LKNK01000010
1280.15865.con.0023 | contig | 24070 bp | LKNK01000006
1280.15865.con.0026 | contig | 31054 bp | LKNK01000009
1280.15865.con.0012 | contig | 107908 bp | LKNK01000002
1280.15865.con.0022 | contig | 257080 bp | LKNK01000005
1280.15865.con.0019 | contig | 4535 bp | LKNK01000026
1280.15865.con.0004 | contig | 147011 bp | LKNK01000012
1280.15865.con.0013 | contig | 24708 bp | LKNK01000020
1280.15865.con.0014 | contig | 39479 bp | LKNK01000021
1280.15865.con.0009 | contig | 93922 bp | LKNK01000017
1280.15865.con.0024 | contig | 319167 bp | LKNK01000007
1280.15865.con.0021 | contig | 87755 bp | LKNK01000004
1280.15865.con.0005 | contig | 449732 bp | LKNK01000013
1280.15865.con.0007 | contig | 20729 bp | LKNK01000015
1280.15865.con.0017 | contig | 3320 bp | LKNK01000024
1280.

## Step 8 — Identify the Missing Contig

The genome metadata reports 26 contigs, while the retrieved sequence records
contain 25 contigs.

Inspection of the sequence identifiers shows that contig `con.0016` is absent
from the retrieved records. The difference between the reported genome length
and the retrieved sequence length is 18,304 bp.

The missing contig is therefore investigated directly through the BV-BRC
Genome Sequence API before proceeding with bulk retrieval.

In [13]:
# Check whether contig 0016 exists in BV-BRC

missing_contig_id = f"{genome_id}.con.0016"

missing_url = (
    "https://www.bv-brc.org/api/genome_sequence/"
    f"?eq(sequence_id,{missing_contig_id})"
    "&http_accept=application/json"
)

missing_response = requests.get(missing_url, timeout=60)

print("HTTP status:", missing_response.status_code)

missing_response.raise_for_status()

missing_record = missing_response.json()

print("Number of records returned:", len(missing_record))

if missing_record:
    print("\nMissing contig record:")
    print(missing_record[0])
else:
    print("\nContig 0016 was not returned by the API.")

HTTP status: 200
Number of records returned: 1

Missing contig record:
{'accession': 'LKNK01000023', 'date_inserted': '2018-05-25T20:40:12.979Z', 'date_modified': '2018-05-25T20:40:12.979Z', 'description': 'LKNK01000023.1', 'gc_content': 31.75, 'genome_id': '1280.15865', 'genome_name': 'Staphylococcus aureus strain N32_250215', 'length': 18304, 'owner': 'PATRIC@patricbrc.org', 'public': True, 'release_date': '2018-01-29T00:00:00Z', 'sequence_id': '1280.15865.con.0016', 'sequence_type': 'contig', 'taxon_id': 1280, 'topology': 'linear', 'version': 1, '_version_': 1809221725678731300, 'sequence': 'taataaataccactaatctaaattaacgaggtgccttatgtataaaaattataacatgacccaacttacactaccaagagaaacttctgttagaatccctcaaagtggtatttcgcgatatgttaatgaaattgttgaaacgatacctgatagcgaattcgatgaattcagacatcatcgtggcgcaacatgctatcatccaataatgatgttaaaaatcatcttatatgcatatactcaatctgtattttctggtcgtagaatagagaaattacttcatgacagtattcgaatgatgtggttagctcaaaatcaaacaccttcttataaaattattaatcgttttagagtaaatcctaatactgatgcattaattgaatctttatttatccagtttc

## Step 8 — Resolve the Sequence-Length Discrepancy

The retrieved FASTA contained 25 sequence records, while BV-BRC metadata
reported 26 contigs.

The missing sequence record was identified as contig `1280.15865.con.0016`.
BV-BRC returned this contig successfully through a direct sequence-record query,
with a length of 18,304 bp.

This exactly matches the observed difference between the reported genome length
and the retrieved FASTA length:

2,730,771 bp − 2,712,467 bp = 18,304 bp

Therefore, the discrepancy is explained by the missing `con.0016` sequence
record in the initial FASTA response.

This validation demonstrates why genome sequence retrieval must be checked
against BV-BRC metadata before performing bulk retrieval.

In [14]:
reported_length = genome_record["genome_length"]
retrieved_length = len(sequence)
missing_length = missing_record[0]["length"]

print("Reported genome length:", reported_length)
print("Retrieved FASTA length:", retrieved_length)
print("Missing contig length:", missing_length)

print("\nLength discrepancy:", reported_length - retrieved_length)
print("Missing contig length:", missing_length)

print(
    "\nDiscrepancy explained:",
    reported_length - retrieved_length == missing_length
)

Reported genome length: 2730771
Retrieved FASTA length: 2712467
Missing contig length: 18304

Length discrepancy: 18304
Missing contig length: 18304

Discrepancy explained: True


## Step 9 — Validate the BV-BRC FTP Genome File

BV-BRC provides genome assemblies through its FTP site, with a separate
directory for each Genome ID.

The `.fna` file contains the nucleotide FASTA sequences of the genome's
contigs.

The FTP version of the test genome is downloaded and compared with the
sequence information previously retrieved through the BV-BRC API.

This provides an independent validation of the genome sequence source before
performing bulk retrieval.

In [16]:
from pathlib import Path
import requests

ftp_url = (
    f"https://ftp.bv-brc.org/genomes/"
    f"{genome_id}/{genome_id}.fna"
)

print("Testing FTP file:")
print(ftp_url)

ftp_response = requests.get(ftp_url, timeout=120)

print("\nHTTP status:", ftp_response.status_code)
print("Downloaded bytes:", len(ftp_response.content))

Testing FTP file:
https://ftp.bv-brc.org/genomes/1280.15865/1280.15865.fna


ConnectTimeout: HTTPSConnectionPool(host='ftp.bv-brc.org', port=443): Max retries exceeded with url: /genomes/1280.15865/1280.15865.fna (Caused by ConnectTimeoutError(<HTTPSConnection(host='ftp.bv-brc.org', port=443) at 0x16a822039d0>, 'Connection to ftp.bv-brc.org timed out. (connect timeout=120)'))

## Step 9 — Validate Complete FASTA Retrieval

The initial FASTA query returned 25 sequence records for a genome reported
to contain 26 contigs.

A direct query for the apparently missing contig confirmed that
`1280.15865.con.0016` exists in BV-BRC and has a length of 18,304 bp.

Because BV-BRC query endpoints support explicit result limits, the FASTA
retrieval is repeated with a sufficiently large limit to ensure that all
sequence records associated with the genome are returned.

The complete retrieved sequence is then compared with the genome-level
metadata.

In [17]:
# Retrieve all contigs for the test genome

complete_fasta_url = (
    f"https://www.bv-brc.org/api/genome_sequence/"
    f"?eq(genome_id,{genome_id})"
    f"&limit(100)"
    f"&http_accept=application/dna+fasta"
)

complete_response = requests.get(
    complete_fasta_url,
    timeout=120
)

print("HTTP status:", complete_response.status_code)

complete_response.raise_for_status()

complete_fasta = complete_response.text

print("FASTA size:", len(complete_fasta), "characters")

HTTP status: 200
FASTA size: 2778716 characters


In [18]:
# Validate the complete FASTA

complete_lines = complete_fasta.splitlines()

complete_header_count = sum(
    line.startswith(">")
    for line in complete_lines
)

complete_sequence = "".join(
    line.strip()
    for line in complete_lines
    if not line.startswith(">")
)

print("Number of FASTA records:", complete_header_count)
print("Total sequence length:", len(complete_sequence))
print("Expected genome length:", genome_record["genome_length"])
print(
    "Length difference:",
    len(complete_sequence) - genome_record["genome_length"]
)

Number of FASTA records: 26
Total sequence length: 2730771
Expected genome length: 2730771
Length difference: 0


In [19]:
# Save the validated complete FASTA

with open(fasta_path, "w") as f:
    f.write(complete_fasta)

print("Updated FASTA:", fasta_path)
print("File size:", fasta_path.stat().st_size, "bytes")

Updated FASTA: ..\data\genomes\1280.15865.fna
File size: 2824269 bytes


In [20]:
# Final validation of the saved FASTA file

with open(fasta_path, "r") as f:
    saved_fasta = f.read()

saved_lines = saved_fasta.splitlines()

saved_header_count = sum(
    line.startswith(">")
    for line in saved_lines
)

saved_sequence = "".join(
    line.strip()
    for line in saved_lines
    if not line.startswith(">")
)

print("Saved FASTA records:", saved_header_count)
print("Saved sequence length:", len(saved_sequence))
print("Expected genome length:", genome_record["genome_length"])
print(
    "Length difference:",
    len(saved_sequence) - genome_record["genome_length"]
)

Saved FASTA records: 26
Saved sequence length: 2730771
Expected genome length: 2730771
Length difference: 0


## Step 10 — Load Genome Cohort

The curated ciprofloxacin dataset contains one record per genome, with
each genome assigned a binary AMR label.

The Genome IDs in this dataset define the cohort for which nucleotide
sequences must be retrieved.

Before downloading sequences, the cohort is loaded and validated to ensure
that Genome IDs are unique and that the expected number of genomes is present.

In [21]:
import pandas as pd

# Load the curated genome-level dataset

data_path = "../data/processed/ciprofloxacin_genome_level.csv"

cohort_df = pd.read_csv(data_path)

print("Dataset shape:", cohort_df.shape)
print("\nColumns:")
print(cohort_df.columns.tolist())

print("\nUnique Genome IDs:", cohort_df["Genome ID"].nunique())
print("\nPhenotype counts:")
print(cohort_df["Phenotype"].value_counts())

print("\nLabel counts:")
print(cohort_df["Label"].value_counts())

Dataset shape: (1205, 3)

Columns:
['Genome ID', 'Phenotype', 'Label']

Unique Genome IDs: 1205

Phenotype counts:
Phenotype
('Susceptible',)    691
('Resistant',)      514
Name: count, dtype: int64

Label counts:
Label
0    691
1    514
Name: count, dtype: int64


## Step 11 — Validate Genome IDs

Genome IDs are used as the primary identifiers for retrieving nucleotide
sequences from BV-BRC.

Each identifier is expected to follow the BV-BRC taxon.version format.
The identifiers are validated before sequence retrieval begins.

In [22]:
import re

# Validate BV-BRC Genome ID format

genome_ids = cohort_df["Genome ID"].astype(str)

valid_id_pattern = genome_ids.str.fullmatch(r"\d+\.\d+")

print("Total Genome IDs:", len(genome_ids))
print("Valid Genome IDs:", valid_id_pattern.sum())
print("Invalid Genome IDs:", (~valid_id_pattern).sum())

if (~valid_id_pattern).sum() > 0:
    print("\nInvalid IDs:")
    print(genome_ids[~valid_id_pattern].tolist())
else:
    print("\nAll Genome IDs have a valid BV-BRC format.")

Total Genome IDs: 1205
Valid Genome IDs: 1205
Invalid Genome IDs: 0

All Genome IDs have a valid BV-BRC format.


## Step 12 — Prepare Genome Output Directory

Genome sequences will be stored as individual FASTA files in the
`data/genomes/` directory.

The directory is created if it does not already exist. Existing FASTA files
are preserved so that the retrieval pipeline can later be made resumable.

In [23]:
from pathlib import Path

# Directory for downloaded genome FASTA files

genome_dir = Path("../data/genomes")
genome_dir.mkdir(parents=True, exist_ok=True)

print("Genome directory:", genome_dir)
print("Directory exists:", genome_dir.exists())

Genome directory: ..\data\genomes
Directory exists: True


## Step 13 — Define Genome Retrieval Function

A dedicated retrieval function is used to download the complete nucleotide
sequence associated with a BV-BRC Genome ID.

The function:
- queries the BV-BRC genome sequence API;
- explicitly sets a large result limit to avoid incomplete retrieval;
- returns the FASTA content;
- raises an error if the API request fails.

Bulk downloading will be performed only after this function has been tested
on individual genomes.

In [24]:
import requests

def retrieve_genome_fasta(genome_id):
    """
    Retrieve the complete genome FASTA for a BV-BRC Genome ID.
    """

    fasta_url = (
        "https://www.bv-brc.org/api/genome_sequence/"
        f"?eq(genome_id,{genome_id})"
        "&limit(1000)"
        "&http_accept=application/dna+fasta"
    )

    response = requests.get(
        fasta_url,
        timeout=120
    )

    response.raise_for_status()

    return response.text

## Step 14 — Test Genome Retrieval

The retrieval function is tested using the representative genome
`1280.15865`, whose sequence was previously validated against BV-BRC
genome metadata.

The retrieved FASTA is checked for the expected number of sequence
records and total nucleotide length.

In [25]:
# Test the retrieval function

test_genome_id = "1280.15865"

test_fasta = retrieve_genome_fasta(test_genome_id)

test_lines = test_fasta.splitlines()

test_header_count = sum(
    line.startswith(">")
    for line in test_lines
)

test_sequence = "".join(
    line.strip()
    for line in test_lines
    if not line.startswith(">")
)

print("Genome ID:", test_genome_id)
print("FASTA records:", test_header_count)
print("Sequence length:", len(test_sequence))

Genome ID: 1280.15865
FASTA records: 26
Sequence length: 2730771


## Step 15 — Validate Retrieved FASTA

Each retrieved FASTA is validated before being saved locally.

The validation checks that the FASTA contains sequence records and that the
total nucleotide sequence is non-empty.

Additional validation against BV-BRC genome metadata will be performed
during bulk retrieval.

In [26]:
def validate_fasta(fasta_text):
    """
    Validate basic FASTA structure and return sequence statistics.
    """

    lines = fasta_text.splitlines()

    header_count = sum(
        line.startswith(">")
        for line in lines
    )

    sequence = "".join(
        line.strip()
        for line in lines
        if not line.startswith(">")
    )

    if header_count == 0:
        raise ValueError("FASTA contains no sequence records.")

    if len(sequence) == 0:
        raise ValueError("FASTA contains no nucleotide sequence.")

    return {
        "records": header_count,
        "length": len(sequence)
    }

In [27]:
# Test the FASTA validation function

test_stats = validate_fasta(test_fasta)

print("FASTA records:", test_stats["records"])
print("Sequence length:", test_stats["length"])

FASTA records: 26
Sequence length: 2730771


## Step 17 — Retrieve Genome Metadata

For each Genome ID, BV-BRC genome metadata provides reference information
such as the expected genome length and number of contigs.

These metadata values will be used to validate the corresponding retrieved
FASTA sequence during bulk genome retrieval.

In [28]:
def retrieve_genome_metadata(genome_id):
    """
    Retrieve genome metadata from BV-BRC.
    """

    metadata_url = (
        "https://www.bv-brc.org/api/genome/"
        f"{genome_id}"
    )

    response = requests.get(
        metadata_url,
        timeout=60
    )

    response.raise_for_status()

    records = response.json()

    if not records:
        raise ValueError(
            f"No metadata found for Genome ID: {genome_id}"
        )

    return records[0]

In [31]:
def retrieve_genome_metadata(genome_id):
    """
    Retrieve genome metadata from BV-BRC.
    """

    metadata_url = (
        "https://www.bv-brc.org/api/genome/"
        f"?eq(genome_id,{genome_id})"
        "&limit(1)"
    )

    response = requests.get(
        metadata_url,
        timeout=60
    )

    response.raise_for_status()

    records = response.json()

    if not records:
        raise ValueError(
            f"No metadata found for Genome ID: {genome_id}"
        )

    return records[0]

## Step 18 — Bulk Genome Retrieval

The complete curated genome cohort is retrieved from BV-BRC.

For each Genome ID, the pipeline:
1. retrieves BV-BRC genome metadata;
2. retrieves the complete FASTA sequence;
3. validates the FASTA structure;
4. compares the retrieved sequence length and contig count with BV-BRC metadata;
5. saves the validated FASTA locally;
6. records the retrieval status.

Existing FASTA files are skipped, allowing the pipeline to resume safely
after interruption.

In [29]:
import time

# Bulk genome retrieval

results = []

for i, genome_id in enumerate(genome_ids, start=1):

    fasta_path = genome_dir / f"{genome_id}.fna"

    # Skip genomes that have already been downloaded
    if fasta_path.exists() and fasta_path.stat().st_size > 0:
        results.append({
            "Genome ID": genome_id,
            "Status": "Already exists"
        })

        print(f"[{i}/{len(genome_ids)}] {genome_id} — already exists")
        continue

    try:
        # Retrieve metadata
        metadata = retrieve_genome_metadata(genome_id)

        expected_length = metadata["genome_length"]
        expected_contigs = metadata["contigs"]

        # Retrieve FASTA
        fasta_text = retrieve_genome_fasta(genome_id)

        # Validate FASTA structure
        stats = validate_fasta(fasta_text)

        retrieved_length = stats["length"]
        retrieved_contigs = stats["records"]

        # Validate against BV-BRC metadata
        if retrieved_length != expected_length:
            raise ValueError(
                f"Length mismatch: "
                f"retrieved={retrieved_length}, "
                f"expected={expected_length}"
            )

        if retrieved_contigs != expected_contigs:
            raise ValueError(
                f"Contig mismatch: "
                f"retrieved={retrieved_contigs}, "
                f"expected={expected_contigs}"
            )

        # Save validated FASTA
        with open(fasta_path, "w") as f:
            f.write(fasta_text)

        results.append({
            "Genome ID": genome_id,
            "Status": "Downloaded",
            "Records": retrieved_contigs,
            "Sequence Length": retrieved_length
        })

        print(
            f"[{i}/{len(genome_ids)}] "
            f"{genome_id} — downloaded and validated"
        )

        # Small delay between requests
        time.sleep(0.2)

    except Exception as e:

        results.append({
            "Genome ID": genome_id,
            "Status": "Failed",
            "Error": str(e)
        })

        print(
            f"[{i}/{len(genome_ids)}] "
            f"{genome_id} — FAILED: {e}"
        )

[1/1205] 1280.15865 — already exists
[2/1205] 1280.15866 — FAILED: 0
[3/1205] 1280.15867 — FAILED: 0
[4/1205] 1280.15868 — FAILED: 0
[5/1205] 1280.15869 — FAILED: 0
[6/1205] 1280.1587 — FAILED: 404 Client Error: Not Found for url: https://www.bv-brc.org/api/genome/1280.1587
[7/1205] 1280.15871 — FAILED: 0
[8/1205] 1280.15872 — FAILED: 0
[9/1205] 1280.15873 — FAILED: 0
[10/1205] 1280.15874 — FAILED: 0
[11/1205] 1280.15875 — FAILED: 0
[12/1205] 1280.15876 — FAILED: 0
[13/1205] 1280.15877 — FAILED: 0
[14/1205] 1280.15878 — FAILED: 0
[15/1205] 1280.1588 — FAILED: 404 Client Error: Not Found for url: https://www.bv-brc.org/api/genome/1280.1588
[16/1205] 1280.15881 — FAILED: 0
[17/1205] 1280.15882 — FAILED: 0
[18/1205] 1280.15883 — FAILED: 0
[19/1205] 1280.15884 — FAILED: 0
[20/1205] 1280.15885 — FAILED: 0
[21/1205] 1280.15886 — FAILED: 0
[22/1205] 1280.15887 — FAILED: 0
[23/1205] 1280.15888 — FAILED: 0
[24/1205] 1280.15889 — FAILED: 0
[25/1205] 1280.1589 — FAILED: 404 Client Error: Not Foun

In [30]:
# Summarize bulk retrieval results

results_df = pd.DataFrame(results)

print("Total processed:", len(results_df))

print("\nStatus counts:")
print(results_df["Status"].value_counts())

print("\nFirst 10 failed genomes:")
print(
    results_df[results_df["Status"] == "Failed"]
    [["Genome ID", "Error"]]
    .head(10)
)

Total processed: 1205

Status counts:
Status
Failed            1204
Already exists       1
Name: count, dtype: int64

First 10 failed genomes:
     Genome ID                                              Error
1   1280.15866                                                  0
2   1280.15867                                                  0
3   1280.15868                                                  0
4   1280.15869                                                  0
5    1280.1587  404 Client Error: Not Found for url: https://w...
6   1280.15871                                                  0
7   1280.15872                                                  0
8   1280.15873                                                  0
9   1280.15874                                                  0
10  1280.15875                                                  0


In [32]:
# Test metadata retrieval

test_metadata = retrieve_genome_metadata("1280.15866")

print("Genome ID:", test_metadata["genome_id"])
print("Genome length:", test_metadata["genome_length"])
print("Contigs:", test_metadata["contigs"])
print("Species:", test_metadata["species"])

Genome ID: 1280.15866
Genome length: 2744592
Contigs: 137
Species: Staphylococcus aureus


In [33]:
# Test a Genome ID that previously returned 404

test_404_id = "1280.1588"

try:
    metadata_404_test = retrieve_genome_metadata(test_404_id)

    print("Genome found:")
    print("Genome ID:", metadata_404_test["genome_id"])
    print("Genome length:", metadata_404_test["genome_length"])
    print("Contigs:", metadata_404_test["contigs"])

except Exception as e:
    print("Genome ID:", test_404_id)
    print("Result:", type(e).__name__)
    print("Error:", e)

Genome ID: 1280.1588
Result: ValueError
Error: No metadata found for Genome ID: 1280.1588


In [35]:
# Check whether the genome has sequence records even though
# genome metadata was not found

sequence_test_url = (
    "https://www.bv-brc.org/api/genome_sequence/"
    "?eq(genome_id,1280.1588)"
    "&limit(1000)"
    "&http_accept=application/dna+fasta"
)

sequence_test_response = requests.get(
    sequence_test_url,
    timeout=120
)

print("HTTP status:", sequence_test_response.status_code)
print("Response length:", len(sequence_test_response.text))

if sequence_test_response.status_code == 200:
    print("\nFASTA preview:")
    print(sequence_test_response.text[:500])

HTTP status: 200
Response length: 0

FASTA preview:



In [36]:
# Test another Genome ID that previously failed

test_missing_id = "1280.1589"

try:
    metadata_test = retrieve_genome_metadata(test_missing_id)
    print("Metadata found:")
    print("Genome ID:", metadata_test["genome_id"])
    print("Genome length:", metadata_test["genome_length"])
    print("Contigs:", metadata_test["contigs"])

except Exception as e:
    print("Genome ID:", test_missing_id)
    print("Metadata result:", type(e).__name__)
    print("Error:", e)


sequence_test_url = (
    "https://www.bv-brc.org/api/genome_sequence/"
    f"?eq(genome_id,{test_missing_id})"
    "&limit(1000)"
    "&http_accept=application/dna+fasta"
)

sequence_test_response = requests.get(
    sequence_test_url,
    timeout=120
)

print("\nSequence API:")
print("HTTP status:", sequence_test_response.status_code)
print("Response length:", len(sequence_test_response.text))

if sequence_test_response.status_code == 200:
    print("FASTA records:", sequence_test_response.text.count(">"))

Genome ID: 1280.1589
Metadata result: ValueError
Error: No metadata found for Genome ID: 1280.1589

Sequence API:
HTTP status: 200
Response length: 0
FASTA records: 0


In [37]:
# Test BV-BRC availability for the first 20 Genome IDs

test_ids = genome_ids.head(20).tolist()

availability_results = []

for genome_id in test_ids:
    try:
        metadata = retrieve_genome_metadata(genome_id)

        sequence_url = (
            "https://www.bv-brc.org/api/genome_sequence/"
            f"?eq(genome_id,{genome_id})"
            "&limit(1000)"
            "&http_accept=application/dna+fasta"
        )

        sequence_response = requests.get(
            sequence_url,
            timeout=120
        )

        fasta_records = sequence_response.text.count(">")

        availability_results.append({
            "Genome ID": genome_id,
            "Metadata": "Found",
            "Sequence Records": fasta_records
        })

    except Exception:
        availability_results.append({
            "Genome ID": genome_id,
            "Metadata": "Not found",
            "Sequence Records": 0
        })

availability_df = pd.DataFrame(availability_results)

print(availability_df)

     Genome ID   Metadata  Sequence Records
0   1280.15865      Found                26
1   1280.15866      Found               137
2   1280.15867      Found                33
3   1280.15868      Found                30
4   1280.15869      Found                35
5    1280.1587  Not found                 0
6   1280.15871      Found                64
7   1280.15872      Found                28
8   1280.15873      Found                51
9   1280.15874      Found                33
10  1280.15875      Found                31
11  1280.15876      Found                55
12  1280.15877      Found                53
13  1280.15878      Found               135
14   1280.1588  Not found                 0
15  1280.15881      Found                86
16  1280.15882      Found                85
17  1280.15883      Found                16
18  1280.15884      Found               123
19  1280.15885      Found                70


## Step 21 — Audit BV-BRC Genome Availability

Before bulk sequence retrieval, the complete genome cohort is checked against
BV-BRC genome metadata.

This determines which phenotype-labeled genomes have corresponding records
available in BV-BRC and which Genome IDs are unavailable.

In [38]:
# Audit BV-BRC metadata availability for the complete cohort

all_genome_ids = genome_ids.tolist()

availability_audit = []

batch_size = 100

for start in range(0, len(all_genome_ids), batch_size):
    batch = all_genome_ids[start:start + batch_size]

    genome_list = ",".join(batch)

    metadata_url = (
        "https://www.bv-brc.org/api/genome/"
        f"?in(genome_id,({genome_list}))"
        "&limit(1000)"
    )

    response = requests.get(
        metadata_url,
        timeout=120
    )

    response.raise_for_status()

    records = response.json()

    found_ids = {
        record["genome_id"]
        for record in records
    }

    for genome_id in batch:
        availability_audit.append({
            "Genome ID": genome_id,
            "Metadata Available": genome_id in found_ids
        })

    print(
        f"Checked {min(start + batch_size, len(all_genome_ids))}"
        f"/{len(all_genome_ids)} genomes"
    )

availability_df = pd.DataFrame(availability_audit)

print("\nAvailability summary:")
print(
    availability_df["Metadata Available"]
    .value_counts()
)

Checked 100/1205 genomes
Checked 200/1205 genomes
Checked 300/1205 genomes
Checked 400/1205 genomes
Checked 500/1205 genomes
Checked 600/1205 genomes
Checked 700/1205 genomes
Checked 800/1205 genomes
Checked 900/1205 genomes
Checked 1000/1205 genomes
Checked 1100/1205 genomes
Checked 1200/1205 genomes
Checked 1205/1205 genomes

Availability summary:
Metadata Available
True     1138
False      67
Name: count, dtype: int64


## Step 22 — Save BV-BRC Availability Audit

The BV-BRC availability audit is saved to preserve a reproducible record of
which phenotype-labeled genomes were available for sequence retrieval.

Genomes without corresponding BV-BRC records are retained in the phenotype
dataset but excluded from genome sequence retrieval.

In [39]:
# Save BV-BRC availability audit

audit_path = "../data/processed/bvbrc_availability_audit.csv"

availability_df.to_csv(
    audit_path,
    index=False
)

print("Saved audit:", audit_path)
print("Rows:", len(availability_df))

Saved audit: ../data/processed/bvbrc_availability_audit.csv
Rows: 1205


## Step 23 — Prepare Genome Retrieval Manifest

The curated phenotype dataset is combined with the BV-BRC availability audit
to create the final genome retrieval manifest.

Only genomes with confirmed BV-BRC metadata availability will proceed to
sequence retrieval.

In [41]:
# Ensure Genome IDs are treated as strings

cohort_df["Genome ID"] = cohort_df["Genome ID"].astype(str)
availability_df["Genome ID"] = availability_df["Genome ID"].astype(str)

print("cohort_df Genome ID type:",
      cohort_df["Genome ID"].dtype)

print("availability_df Genome ID type:",
      availability_df["Genome ID"].dtype)

cohort_df Genome ID type: str
availability_df Genome ID type: str


In [42]:
# Create the final genome retrieval manifest

retrieval_df = cohort_df.merge(
    availability_df,
    on="Genome ID",
    how="left"
)

print("Manifest shape:", retrieval_df.shape)

print("\nColumns:")
print(retrieval_df.columns.tolist())

print("\nBV-BRC availability:")
print(
    retrieval_df["Metadata Available"]
    .value_counts()
)

print("\nFirst 5 rows:")
print(retrieval_df.head())

Manifest shape: (1205, 4)

Columns:
['Genome ID', 'Phenotype', 'Label', 'Metadata Available']

BV-BRC availability:
Metadata Available
True     1138
False      67
Name: count, dtype: int64

First 5 rows:
    Genome ID         Phenotype  Label  Metadata Available
0  1280.15865  ('Susceptible',)      0                True
1  1280.15866  ('Susceptible',)      0                True
2  1280.15867  ('Susceptible',)      0                True
3  1280.15868  ('Susceptible',)      0                True
4  1280.15869  ('Susceptible',)      0                True


## Step 24 — Select Retrievable Genomes

Only genomes with confirmed BV-BRC metadata availability are selected for
sequence retrieval.

The original phenotype cohort is retained separately, including genomes that
cannot currently be retrieved from BV-BRC.

In [43]:
# Select genomes available in BV-BRC

retrievable_df = retrieval_df[
    retrieval_df["Metadata Available"] == True
].copy()

print("Retrievable genomes:", len(retrievable_df))
print("Excluded genomes:", len(retrieval_df) - len(retrievable_df))

print("\nLabel distribution:")
print(retrievable_df["Label"].value_counts())

print("\nPhenotype distribution:")
print(retrievable_df["Phenotype"].value_counts())

Retrievable genomes: 1138
Excluded genomes: 67

Label distribution:
Label
0    664
1    474
Name: count, dtype: int64

Phenotype distribution:
Phenotype
('Susceptible',)    664
('Resistant',)      474
Name: count, dtype: int64


## Step 25 — Define Robust Genome Retrieval

A robust retrieval function is used to download and validate each genome
sequence before saving it locally.

For each genome:

1. Retrieve BV-BRC genome metadata.
2. Retrieve all associated FASTA records.
3. Validate the number of contigs.
4. Validate the total nucleotide sequence length.
5. Save the FASTA only if validation succeeds.
6. Record failures without stopping the complete retrieval process.

The procedure is designed to be resumable and reproducible.

In [44]:
def retrieve_and_validate_genome(genome_id, output_dir):
    """
    Retrieve, validate, and save a complete BV-BRC genome FASTA.

    Returns a dictionary describing the retrieval result.
    """

    fasta_path = output_dir / f"{genome_id}.fna"

    try:
        # --------------------------------------------------
        # 1. Retrieve genome metadata
        # --------------------------------------------------
        metadata = retrieve_genome_metadata(genome_id)

        expected_length = metadata["genome_length"]
        expected_contigs = metadata["contigs"]

        # --------------------------------------------------
        # 2. Retrieve complete FASTA
        # --------------------------------------------------
        fasta_text = retrieve_genome_fasta(genome_id)

        # --------------------------------------------------
        # 3. Validate FASTA
        # --------------------------------------------------
        stats = validate_fasta(fasta_text)

        retrieved_length = stats["length"]
        retrieved_contigs = stats["records"]

        # --------------------------------------------------
        # 4. Validate sequence length
        # --------------------------------------------------
        if retrieved_length != expected_length:
            raise ValueError(
                f"Sequence length mismatch: "
                f"expected {expected_length}, "
                f"got {retrieved_length}"
            )

        # --------------------------------------------------
        # 5. Validate contig count
        # --------------------------------------------------
        if retrieved_contigs != expected_contigs:
            raise ValueError(
                f"Contig count mismatch: "
                f"expected {expected_contigs}, "
                f"got {retrieved_contigs}"
            )

        # --------------------------------------------------
        # 6. Save validated FASTA
        # --------------------------------------------------
        with open(fasta_path, "w") as f:
            f.write(fasta_text)

        return {
            "Genome ID": genome_id,
            "Status": "Downloaded",
            "Genome Length": expected_length,
            "Contigs": expected_contigs,
            "File": str(fasta_path),
            "Error": ""
        }

    except Exception as e:

        return {
            "Genome ID": genome_id,
            "Status": "Failed",
            "Genome Length": None,
            "Contigs": None,
            "File": "",
            "Error": str(e)
        }

In [45]:
test_result = retrieve_and_validate_genome(
    "1280.15866",
    genome_dir
)

print(test_result)

{'Genome ID': '1280.15866', 'Status': 'Downloaded', 'Genome Length': 2744592, 'Contigs': 137, 'File': '..\\data\\genomes\\1280.15866.fna', 'Error': ''}


## Step 26 — Bulk Genome Retrieval

All genomes with confirmed BV-BRC availability are retrieved using the validated
retrieval function.

Existing validated FASTA files are skipped, allowing the process to resume
without redownloading completed genomes.

A retrieval manifest records the status and validation results for every genome.

In [46]:
# Bulk retrieval of all available genomes

retrieval_results = []

for i, genome_id in enumerate(
    retrievable_df["Genome ID"],
    start=1
):

    fasta_path = genome_dir / f"{genome_id}.fna"

    # Skip genomes that have already been retrieved
    if fasta_path.exists() and fasta_path.stat().st_size > 0:

        retrieval_results.append({
            "Genome ID": genome_id,
            "Status": "Already exists",
            "Genome Length": None,
            "Contigs": None,
            "File": str(fasta_path),
            "Error": ""
        })

        print(
            f"[{i}/{len(retrievable_df)}] "
            f"{genome_id} — already exists"
        )

        continue

    result = retrieve_and_validate_genome(
        genome_id,
        genome_dir
    )

    retrieval_results.append(result)

    print(
        f"[{i}/{len(retrievable_df)}] "
        f"{genome_id} — {result['Status']}"
    )

[1/1138] 1280.15865 — already exists
[2/1138] 1280.15866 — already exists
[3/1138] 1280.15867 — Downloaded
[4/1138] 1280.15868 — Downloaded
[5/1138] 1280.15869 — Downloaded
[6/1138] 1280.15871 — Downloaded
[7/1138] 1280.15872 — Downloaded
[8/1138] 1280.15873 — Downloaded
[9/1138] 1280.15874 — Downloaded
[10/1138] 1280.15875 — Downloaded
[11/1138] 1280.15876 — Downloaded
[12/1138] 1280.15877 — Downloaded
[13/1138] 1280.15878 — Downloaded
[14/1138] 1280.15881 — Downloaded
[15/1138] 1280.15882 — Downloaded
[16/1138] 1280.15883 — Downloaded
[17/1138] 1280.15884 — Downloaded
[18/1138] 1280.15885 — Downloaded
[19/1138] 1280.15886 — Downloaded
[20/1138] 1280.15887 — Downloaded
[21/1138] 1280.15888 — Downloaded
[22/1138] 1280.15889 — Downloaded
[23/1138] 1280.15891 — Downloaded
[24/1138] 1280.15892 — Downloaded
[25/1138] 1280.15893 — Downloaded
[26/1138] 1280.15894 — Downloaded
[27/1138] 1280.15895 — Downloaded
[28/1138] 1280.15896 — Downloaded
[29/1138] 1280.15897 — Downloaded
[30/1138] 1280.